# Cop & Thief — Results Analysis

Scientific analysis notebook for the Dual-AI-Agent pursuit game (University of Haifa, *Orchestration of AI Agents*, Task 6). Figures are produced reproducibly by `scripts/make_figures.py` and `scripts/sensitivity.py`; this notebook collects the formal model, methodology, results, and references. Re-render figures with `uv run python scripts/sensitivity.py`.

## 1. Formal model — Dec-POMDP

The game is a **Decentralized Partially Observable Markov Decision Process** [1, 2]:

$$\langle n,\, S,\, \{A_i\},\, P,\, R,\, \{O_i\},\, O,\, \gamma \rangle$$

- $n = 2$ agents (Cop, Thief).
- $S$: grid configuration $=$ cop cell, thief cell, and barrier set on a $W\times H$ board.
- $A_i = \{\,\mathrm{MOVE}(dx,dy)\ (dx,dy\in\{-1,0,1\}),\ \mathrm{PLACE\_BARRIER}\ (\text{cop}),\ \mathrm{STAY}\,\}$.
- $P(s'\mid s,a)$: deterministic engine transition (legality + capture check).
- $R$: scoring table — cop win $(20,5)$, thief win $(5,10)$.
- $O_i$: each agent's partial view within `visibility_radius` $+$ inbound natural-language messages.
- $O$: observation function mapping the true state to each agent's partial view.
- $\gamma$: discount factor (used only if the optional Q-learning track is enabled).

Because each agent acts on a *local* observation and free-text messages (no shared coordinate protocol), coordination must **emerge** from natural-language interpretation under uncertainty.

## 2. Decision strategy

Per the assignment, the **move strategy is heuristic** (the LLM handles communication, not movement):

- **Cop** — `heuristic`: minimise Chebyshev distance to the believed thief cell. `smart`: one-ply look-ahead ranking actions by $(\text{distance},\ \text{thief escape-options})$ after the thief's reply, herding the thief into a corner.
- **Thief** — `greedy`: maximise distance (ties → wall-hugging, the *always-left* artefact). `smart` (default): rank by $(\text{distance from cop},\ \text{mobility},\ \text{centrality})$ to flee while keeping escape room.

**Optional RL (design):** a tabular Q-learning policy would update via the Bellman equation [3, 4]

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,\big[\,r + \gamma\,\max_{a'} Q(s',a') - Q(s,a)\,\big]$$

with $\varepsilon$-greedy exploration. Not implemented (heuristic suffices for the graded communication metric); learning-curve plots would be added here if enabled.

## 3. Strategy comparison

Greedy vs cornering cop, and the cop win-rate by strategy (40 seeds, 5×5):

![win rate vs grid size](../assets/winrate_vs_gridsize.png)
![strategy comparison](../assets/heuristic_vs_nl.png)

The greedy cop falls into a limit cycle (~0.72 on 5×5); the cornering cop reaches **1.00 vs the greedy thief**. Against the *smart* thief, capture drops (1.00/1.00/0.90/0.88/0.47 on 3×3–7×7) — a better evader is genuinely harder to corner.

## 4. Sensitivity analysis (OAT)

One-at-a-time sweeps (vary one parameter, fix the rest, average over seeds).

### 4.1 Partial observation (visibility radius)
![capture vs visibility](../assets/sensitivity_visibility.png)
![capture heatmap](../assets/sensitivity_heatmap.png)

At radius 0 (messages only) capture collapses to 0.03–0.20; it peaks at radius 1–2 then declines (a greedy NL cop does not exploit a wider view), and falls as the board grows.

### 4.2 Board size and game length
![moves boxplot](../assets/moves_boxplot.png)
![area vs moves scatter](../assets/scatter_area_moves.png)

Mean moves-to-conclusion grows with board area (2×2≈2 → 7×7≈18); large boards cluster at the 25-move cap (escapes).

## 5. Token cost

Interpretation is one LLM call per turn (~66/match, 5×5); enabling creative speech adds a generation call per turn (~2×). Offline estimate at gpt-4o-mini list price ≈ **$0.0006/match** (see README R.7, `results/token_cost.txt`).

## References

1. D. S. Bernstein, R. Givan, N. Immerman, S. Zilberstein. *The Complexity of Decentralized Control of Markov Decision Processes.* Mathematics of Operations Research, 27(4):819–840, 2002.
2. F. A. Oliehoek, C. Amato. *A Concise Introduction to Decentralized POMDPs.* Springer, 2016.
3. C. J. C. H. Watkins, P. Dayan. *Q-learning.* Machine Learning, 8:279–292, 1992.
4. R. S. Sutton, A. G. Barto. *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press, 2018.
5. J. Nielsen. *Enhancing the explanatory power of usability heuristics.* CHI '94, 152–158.
6. ISO/IEC 25010:2011. *Systems and software engineering — SQuaRE — System and software quality models.*